# Run treatment-anchor full-cohort training

A simple notebook fallback for `launch_full_cohort.sh`. It follows the smoke-test pattern: read the manifest, invoke the existing training module once per event, stream its output, and continue after failures. Events run sequentially, while each sklearn/joblib fit receives `n_jobs=-1` and can use every available CPU.

Run this only in a compute-backed Jupyter session, not on an ERISTwo login node. Existing complete outputs are skipped by the worker unless `OVERWRITE=True`.

In [ ]:
from __future__ import annotations

import subprocess
import sys
import time
from pathlib import Path

from tqdm.auto import tqdm


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.py').is_file() and (candidate / 'pipelines').is_dir():
            return candidate
    raise RuntimeError(f'Could not find v2 root from {start}')


V2_ROOT = find_v2_root()
print(f'Python:  {sys.executable}')
print(f'v2 root: {V2_ROOT}')


In [ ]:
ANCHOR = 'treatment'
N_JOBS = -1
MAX_ITER = 2500
BACKEND = 'threading'
OVERWRITE = False
MAX_TASKS = None  # Set to a small integer for a trial; None runs the full manifest.

MANIFEST = V2_ROOT / 'slurm' / 'slurm_manifests' / 'full_cohort_tasks.tsv'
if ANCHOR != 'treatment':
    raise ValueError('This notebook is restricted to treatment-anchor runs')
if not MANIFEST.is_file():
    raise FileNotFoundError(f'Manifest not found: {MANIFEST}')

print(f'Anchor:   {ANCHOR}')
print(f'Manifest: {MANIFEST}')
print(f'n_jobs:   {N_JOBS} (all available CPUs)')


In [ ]:
SCHEME_ALIASES = {'icd3': 'icd3_post', 'icd4': 'icd4_post', 'phecode': 'phecode_post'}
tasks: list[tuple[str, str]] = []

for line_number, raw in enumerate(MANIFEST.read_text().splitlines(), 1):
    if not raw.strip():
        continue
    fields = raw.split('\t')
    if len(fields) != 2 or not all(fields):
        raise ValueError(f'{MANIFEST}:{line_number}: expected scheme<TAB>event')
    scheme, event = fields
    tasks.append((SCHEME_ALIASES.get(scheme, scheme), event))

if MAX_TASKS is not None:
    tasks = tasks[:MAX_TASKS]

print(f'Tasks: {len(tasks)}')
print('First tasks:', tasks[:5])


In [ ]:
def build_command(scheme: str, event: str) -> list[str]:
    command = [
        sys.executable, '-m', 'pipelines.training.run_full_cohort_event',
        '--scheme', scheme,
        '--event', event,
        '--anchor', ANCHOR,
        '--n-jobs', str(N_JOBS),
        '--max-iter', str(MAX_ITER),
        '--backend', BACKEND,
    ]
    if OVERWRITE:
        command.append('--overwrite')
    return command


results = []
for scheme, event in tqdm(tasks, desc='full cohort'):
    command = build_command(scheme, event)
    print(f'\n=== {scheme}:{event} ===', flush=True)
    started = time.perf_counter()
    process = subprocess.run(command, cwd=V2_ROOT)
    elapsed_minutes = (time.perf_counter() - started) / 60
    result = {
        'scheme': scheme,
        'event': event,
        'returncode': process.returncode,
        'minutes': elapsed_minutes,
    }
    results.append(result)
    print(f"[{scheme}:{event}] exit={process.returncode}, wall={elapsed_minutes:.1f}m")

succeeded = sum(result['returncode'] == 0 for result in results)
failed = [result for result in results if result['returncode'] != 0]
print(f'\nDone. {succeeded} succeeded, {len(failed)} failed.')
for result in failed:
    print(f"  {result['scheme']}:{result['event']} (exit {result['returncode']})")
